In [0]:
%sql
MERGE INTO intellbi_catalog.intellbi_gold.RefinedMonthlySales as tgt
USING (select* from intellbi_catalog.intellbi_silver.cleansedmonthlysales 

WHERE ingest_ts >
(
    SELECT coalesce(max(last_modified_ts), to_timestamp ('1990-01-01','yyyy-MM-dd'))
    from intellbi_catalog.intellbi_gold.RefinedMonthlySales
    ))

as src
ON tgt.sale_id = src.sale_id
WHEN MATCHED AND
tgt.product<> src.product OR
tgt.category<>src.category OR
tgt.quantity<> src.quantity OR
tgt.price<> src.price OR
tgt.sale_date<> src.sale_date OR
tgt.region<> src.region 
THEN UPDATE SET 
tgt.product=src.product,
tgt.category=src.category,
tgt.quantity=src.quantity,
tgt.price=src.price,
tgt.sale_date=src.sale_date,
tgt.region=src.region,
tgt.last_modified_ts=current_timestamp()
WHEN NOT MATCHED
THEN INSERT (sale_id,product,category,quantity,price,sale_date,region,
initial_load_ts,last_modified_ts)
VALUES (src.sale_id,
src.product,src.category,
src.quantity,
src.price,src.sale_date,
src.region,
current_timestamp(),
current_timestamp());



